In [1]:
import pandas as pd
import numpy as np
import re
import ast

In [2]:
import warnings
warnings.filterwarnings('ignore')

# Beko

In [3]:
path1 = "../web-scraping-data/data/raw/beko/beko_urun_ozellik.csv"
path2 = "../web-scraping-data/data/raw/beko/beko_urun_ozellik_2.csv"
path3 = "../web-scraping-data/data/raw/beko/beko_urun_ozellik3.csv"

df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)
df3 = pd.read_csv(path3)

print(f"df1 shape: {df1.shape}")
print(f"df2 shape: {df2.shape}")
print(f"df3 shape: {df3.shape}")


df1 shape: (1184, 10)
df2 shape: (126, 10)
df3 shape: (651, 10)


In [4]:
# beko_turleri.csv dosyasını kat_tur.csv olarak kaydet
beko_turleri_path = "../web-scraping-data/data/raw/beko/beko_turleri.csv"
kat_tur_df = pd.read_csv(beko_turleri_path)

print(kat_tur_df.head())

     kategori          urun_adi
0  Beyaz Eşya         Buzdolabı
1  Beyaz Eşya   Derin Dondurucu
2  Beyaz Eşya  Çamaşır Makinesi
3  Beyaz Eşya  Bulaşık Makinesi
4  Beyaz Eşya  Kurutma Makinesi


In [5]:
# birleştir ve indeksleri sıfırla, aynı ürünü tekilleştir
df = pd.concat([df1, df2, df3], ignore_index=True, sort=False)
df.drop_duplicates(subset='urun_linki', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"merged df shape: {df.shape}")
df.sample(5, random_state=15)

merged df shape: (1301, 10)


,urun_linki,urun_adi,fiyat,teknik_ozellikler,enerji_sinifi,favori_sayisi,yorum_sayisi,renk_secenekleri,urun_ozellikleri,yorumlar
94,https://www.beko.com.tr/12-kg-camasir-makinesi...,CMX 12140 12 Kg Çamaşır Makinesi,39.959 TL,"{""Kapasite"": ""12 kg"", ""Maksimum Sıkma Devri"": ...",A,120.0,NaN,[],"{""Kapasite"": ""12 kg"", ""Maksimum Sıkma Devri"": ...","[{""yildiz"": ""5"", ""baslik"": ""F/P ÜRÜNÜ"", ""sahip..."
482,https://www.beko.com.tr/kulaklik/jr310bt-bluet...,JR310BT Bluetooth ÇocukKulaklığı OEMavi Kulaklık,2.190 TL,"{""Kulaklık Tipi"": ""Kablosuz kulak üstü kulaklı...",NaN,0.0,NaN,[],"{""Kulaklık Tipi"": ""Kablosuz kulak üstü kulaklı...",[]
295,https://www.beko.com.tr/fitfry-buhar-destekli-...,BFC 450 A FitFry Buhar Destekli Fırın,59.405 TL,"{""Motor Teknolojisi"": ""AeroPerfect"", ""Homewhiz...",A+,5.0,NaN,[],"{""Motor Teknolojisi"": ""AeroPerfect"", ""Homewhiz...",[]
434,https://www.beko.com.tr/hoparlor/band-bt-hopar...,Grundig Band BT Hoparlör Yeşil Hoparlör,2.099 TL,"{""Ses Çıkış Gücü"": ""2x 2,5 W"", ""Çalma Süresi"":...",NaN,2.0,NaN,[],"{""Ses Çıkış Gücü"": ""2x 2,5 W"", ""Çalma Süresi"":...",[]
1015,https://www.beko.com.tr/kucuk-ev-aletleri-yede...,Bıçak Grubu Blender Küçük Ev Aletleri Yedek Pa...,475 TL,{},NaN,1.0,1,[],{},[]


In [7]:
df = df.drop_duplicates().reset_index(drop=True)
df = df.drop_duplicates(subset='urun_linki', keep='first').reset_index(drop=True)
print(f"df shape after removing duplicates: {df.shape}")

df shape after removing duplicates: (1301, 10)


In [8]:
df_yedek = df.copy()

In [9]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   urun_linki         1301 non-null   str    
 1   urun_adi           1301 non-null   str    
 2   fiyat              1301 non-null   str    
 3   teknik_ozellikler  1301 non-null   str    
 4   enerji_sinifi      270 non-null    object 
 5   favori_sayisi      1237 non-null   float64
 6   yorum_sayisi       126 non-null    object 
 7   renk_secenekleri   1301 non-null   str    
 8   urun_ozellikleri   1301 non-null   str    
 9   yorumlar           1301 non-null   str    
dtypes: float64(1), object(2), str(7)
memory usage: 1.2+ MB


In [10]:
# 1. Kategori Sütununu Çıkarma
# URL'i '/' işaretine göre böldüğümüzde liste şu şekilde olur: 
# ['https:', '', 'www.beko.com.tr', 'kategori_adi', 'urun_adi']
# Bize 3. indeksteki 'kategori_adi' lazım.
df['Product_Group'] = df['urun_linki'].str.split('/').str[3]

# 2. Tire (-) İşaretlerini Temizleme
# Tireleri boşluk karakteri ile değiştiriyoruz
df['Product_Group'] = df['Product_Group'].str.replace('-', ' ', regex=False)

# 3. Metin Formatını Düzeltme
# Her kelimenin ilk harfini büyük yapmak için (Örn: "oyun konsolu" -> "Oyun Konsolu")
df['Product_Group'] = df['Product_Group'].str.title()

# Sonucu kontrol etmek için
print(df[['urun_linki', 'Product_Group']].sample(5, random_state=15))

                                             urun_linki  \
94    https://www.beko.com.tr/12-kg-camasir-makinesi...   
482   https://www.beko.com.tr/kulaklik/jr310bt-bluet...   
295   https://www.beko.com.tr/fitfry-buhar-destekli-...   
434   https://www.beko.com.tr/hoparlor/band-bt-hopar...   
1015  https://www.beko.com.tr/kucuk-ev-aletleri-yede...   

                                      Product_Group  
94                           12 Kg Camasir Makinesi  
482                                        Kulaklik  
295                     Fitfry Buhar Destekli Firin  
434                                        Hoparlor  
1015  Kucuk Ev Aletleri Yedek Parca Ve Aksesuarlari  


In [11]:
unique_values = pd.Series(df["Product_Group"].dropna().unique()).sort_values()
print(unique_values.to_string(index=False))

                       10 Kg Camasir Makinesi
                       10 Kg Kurutma Makinesi
                       11 Kg Camasir Makinesi
                       11 Kg Kurutma Makinesi
                       12 Kg Camasir Makinesi
                       12 Kg Kurutma Makinesi
                    3 Bolmeli Derin Dondurucu
                 3 Programli Bulasik Makinesi
                    4 Bolmeli Derin Dondurucu
                 4 Programli Bulasik Makinesi
                    5 Bolmeli Derin Dondurucu
                 5 Programli Bulasik Makinesi
                    6 Bolmeli Derin Dondurucu
                 6 Programli Bulasik Makinesi
                    7 Bolmeli Derin Dondurucu
                   8 Bolmeli Derin Dondurucu 
                        8 Kg Camasir Makinesi
                        8 Kg Kurutma Makinesi
                        9 Kg Camasir Makinesi
                        9 Kg Kurutma Makinesi
                           Ada Tipi Davlumbaz
                                  

In [12]:

# Türkçe karakterleri İngilizce karşılıklarına çeviren ve küçük harf yapan fonksiyon
def metin_temizle(text):
    if pd.isna(text):
        return ""
    text = str(text).lower() 
    ceviri_tablosu = str.maketrans("çğıöşü", "cgiosu")
    text = text.translate(ceviri_tablosu)
    return " ".join(text.split())

# 1. Aşama: Senin Manuel Eşleştirmelerin ve Otomatik Eşleşmelerin Sözlüğü
# Sözlük Yapısı: {'Alt Kategori (Temizlenmiş)': ('Ana Kategori', 'Orijinal Alt Kategori Adı')}
kategori_kurallari = {
    # BEYAZ EŞYA GRUBU
    'camasir makinesi': ('Beyaz Eşya', 'Çamaşır Makinesi'),
    'kurutma makinesi': ('Beyaz Eşya', 'Kurutma Makinesi'),
    'kurutmali camasir makinesi': ('Beyaz Eşya', 'Kurutmalı Çamaşır Makinesi'),
    'bulasik makinesi': ('Beyaz Eşya', 'Bulaşık Makinesi'),
    'derin dondurucu': ('Beyaz Eşya', 'Derin Dondurucu'),
    'buzdolabi': ('Beyaz Eşya', 'Buzdolabı'),
    'firin': ('Beyaz Eşya', 'Fırın'),
    'su sebili': ('Beyaz Eşya', 'Su Sebili'),
    'beyaz esya yedek parca': ('Beyaz Eşya', 'Beyaz Eşya Yedek Parça ve Aksesuarları'),
    'mikrodalga': ('Beyaz Eşya', 'Mikrodalga Fırın'), # Senin Kuralın
    'set ustu ocak': ('Beyaz Eşya', 'Set Üstü Ocak'), # Senin Kuralın
    'domino ocak': ('Beyaz Eşya', 'Set Üstü Ocak'),
    'gazli ocak': ('Beyaz Eşya', 'Set Üstü Ocak'),
    'induksiyonlu ocak': ('Beyaz Eşya', 'Set Üstü Ocak'),
    'vitroseramik ocak': ('Beyaz Eşya', 'Set Üstü Ocak'),

    # ANKASTRE GRUBU
    'ankastre bulasik': ('Ankastre', 'Ankastre Bulaşık Makineleri'), # Senin Kuralın
    'ankastre buzdolabi': ('Ankastre', 'Ankastre Buzdolabı'),
    'ankastre firin': ('Ankastre', 'Ankastre Fırın'),
    'ankastre mikrodalga': ('Ankastre', 'Ankastre Mikrodalgalar'),
    'ankastre davlumbaz': ('Ankastre', 'Ankastre Davlumbazlar'),
    'ankastre aspirator': ('Ankastre', 'Ankastre Aspiratörler'),
    'ankastre set': ('Ankastre', 'Ankastre Set'), # Senin Kuralın
    'ankastre ocak': ('Ankastre', 'Ankastre Ocak'), # Senin Kuralın
    'pisirici yedek parca': ('Ankastre', 'Pişirici Yedek Parça ve Aksesuarları'),
    'ada tipi davlumbaz': ('Ankastre', 'Ankastre Davlumbazlar'),
    'duvar tipi davlumbaz': ('Ankastre', 'Ankastre Davlumbazlar'),
    'gomme aspirator': ('Ankastre', 'Ankastre Aspiratörler'),

    # ELEKTRONİK GRUBU
    'cep telefonu': ('Elektronik', 'Cep Telefonu'), # Senin Kuralın
    'iphone': ('Elektronik', 'Cep Telefonu'),
    'android': ('Elektronik', 'Cep Telefonu'),
    'giyilebilir': ('Elektronik', 'Giyilebilir Teknoloji'), # Senin Kuralın
    'akilli saat': ('Elektronik', 'Giyilebilir Teknoloji'),
    'akilli bileklik': ('Elektronik', 'Giyilebilir Teknoloji'),
    'bilgisayar': ('Elektronik', 'Bilgisayar'), # Senin Kuralın
    'laptop': ('Elektronik', 'Bilgisayar'),
    'tablet': ('Elektronik', 'Bilgisayar'),
    'ses goruntu': ('Elektronik', 'Ses& Görüntü Sistemleri'), # Senin Kuralın
    'hoparlor': ('Elektronik', 'Ses& Görüntü Sistemleri'),
    'kulaklik': ('Elektronik', 'Ses& Görüntü Sistemleri'),
    'oyun': ('Elektronik', 'Hobi - Oyun (Gaming) Ürünleri'), # Senin Kuralın
    'konsol': ('Elektronik', 'Hobi - Oyun (Gaming) Ürünleri'),
    'arac ici kamera': ('Elektronik', 'Araç İçi Kamera'), # Senin Kuralın
    'yazar kasa': ('Elektronik', 'Ödeme Sistemleri'), # Senin Kuralın
    'pos': ('Elektronik', 'Ödeme Sistemleri'),
    'cep telefonu aksesuarlari': ('Elektronik', 'Cep Telefonu Aksesuarları'),
    'powerbank': ('Elektronik', 'Cep Telefonu Aksesuarları'),
    'sarj cihazi': ('Elektronik', 'Cep Telefonu Aksesuarları'),
    
    # ISITMA SOĞUTMA VE ENERJİ GRUBU
    'klima': ('Isıtma Soğutma ve Enerji', 'Klima'),
    'vantilator': ('Isıtma Soğutma ve Enerji', 'Vantilatör'),
    'isitici': ('Elektronik', 'Elektrikli Isıtıcı'), # Senin Kuralın
    'radyator': ('Elektronik', 'Elektrikli Isıtıcı'), 
    'termosifon': ('Isıtma Soğutma ve Enerji', 'Termosifon'),
    'kombi': ('Isıtma Soğutma ve Enerji', 'Kombi'),
    'hava sogutucu': ('Isıtma Soğutma ve Enerji', 'Hava Soğutucu'),
    'nem alma': ('Isıtma Soğutma ve Enerji', 'Nem Alma Cihazı'),
    'isitma sogutma yedek parca': ('Isıtma Soğutma ve Enerji', 'Isıtma Soğutma Yedek Parça ve Aksesuarları'), # Senin Kuralın

    # KÜÇÜK EV ALETLERİ GRUBU
    'kucuk ev aletleri': ('Küçük Ev Aletleri', 'Küçük Ev Aletleri Seti'), # Senin Kuralın (Setleri)
    'supurge': ('Küçük Ev Aletleri', 'Elektrikli Süpürge'), # Senin Kuralın
    'frozen': ('Küçük Ev Aletleri', 'Frozen ve Slush Makinesi'), # Senin Kuralın
    'slush': ('Küçük Ev Aletleri', 'Frozen ve Slush Makinesi'),
    'kahve makinesi': ('Küçük Ev Aletleri', 'Kahve Makinesi'), # Senin Kuralın
    'espresso': ('Küçük Ev Aletleri', 'Kahve Makinesi'),
    'pisirici': ('Küçük Ev Aletleri', 'Pişirici'), # Senin Kuralın
    'airfryer': ('Küçük Ev Aletleri', 'Pişirici'),
    'fritoz': ('Küçük Ev Aletleri', 'Pişirici'),
    'tost makinesi': ('Küçük Ev Aletleri', 'Pişirici'),
    'ekmek kizartma': ('Küçük Ev Aletleri', 'Pişirici'),
    'karistirici': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'), # Senin Kuralın
    'dograyici': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'blender': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'mikser': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'kiyma makinesi': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'hamur yogurma': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'kisisel bakim': ('Küçük Ev Aletleri', 'Kişisel Bakım'), # Senin Kuralın
    'agiz bakim': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'baskul': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'sac duzlestirici': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'sac masasi': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'tiras makinesi': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'outdoor': ('Küçük Ev Aletleri', 'Outdoor Ekipman'), # Senin Kuralın
    'termos': ('Küçük Ev Aletleri', 'Outdoor Ekipman'),
    'kahve ve sofra': ('Küçük Ev Aletleri', 'Kahve ve Sofra Ürünleri'), # Senin Kuralın
    'ev temizlik': ('Küçük Ev Aletleri', 'Ev Temizlik Ekipmanları ve Teknolojileri'), # Senin Kuralın
    'utu': ('Küçük Ev Aletleri', 'Ütü'),
    'cay makinesi': ('Küçük Ev Aletleri', 'Çay Makinesi'),
    'semaver': ('Küçük Ev Aletleri', 'Semaver'),
    'kettle': ('Küçük Ev Aletleri', 'Kettle'),
    'meyve sikacagi': ('Küçük Ev Aletleri', 'Katı Meyve Sıkacağı'),
    'narenciye': ('Küçük Ev Aletleri', 'Narenciye Sıkacağı'),
    'dondurma makinesi': ('Küçük Ev Aletleri', 'Dondurma Makinesi'),
    'buz makinesi': ('Küçük Ev Aletleri', 'Buz Makinesi'),
    'uv temizleme': ('Küçük Ev Aletleri', 'UV Temizleme Cihazı'),
    'yogurt': ('Küçük Ev Aletleri', 'Yoğurt &Kefir Makinesi'),
    'kefir': ('Küçük Ev Aletleri', 'Yoğurt &Kefir Makinesi'),

    # HİJYEN-AKSESUAR-OTO GRUBU
    'aktif hijyen': ('Hijyen-Aksesuar-Oto', 'Aktif Hijyen'), # Senin Kuralın
    'arac buzdolabi': ('Hijyen-Aksesuar-Oto', 'Araç Buzdolabı'),
    'aksesuar': ('Hijyen-Aksesuar-Oto', 'Aksesuarlar'),

    # SU ARITMA GRUBU
    'su aritma': ('Su Arıtma', 'Su Arıtma Cihazları') # Senin Kuralın
}

# 2. Aşama: Eşleştirme Fonksiyonu
def akilli_eslestir(Product_Group):
    # Ürün adını temizle (Örn: "10 Kg Camasir Makinesi" -> "10 kg camasir makinesi")
    temiz_ad = metin_temizle(Product_Group)
    
    # Kural sözlüğündeki anahtar kelimeleri (keys) tek tek dolaş
    for anahtar_kelime, (ana_kat, alt_kat) in kategori_kurallari.items():
        # Eğer sözlükteki anahtar kelime, temizlenmiş ürün adının içinde geçiyorsa
        # (Örn: "camasir makinesi" metni "10 kg camasir makinesi" içinde geçer)
        if re.search(r'\b' + re.escape(anahtar_kelime) + r'\b', temiz_ad):
            return pd.Series([ana_kat, alt_kat])
            
    # Eğer hiçbir kurala uymazsa
    return pd.Series(['Diğer', 'Diğer'])

# 3. Aşama: Fonksiyonu Uygulama ve Yeni Sütunları Oluşturma
# df'indeki 'kategori' sütununu baz alıyoruz (veya doğrudan 'Product_Group' sütununu da kullanabilirsin)
# Eğer senin scraping verinde 'Product_Group' daha açıklayıcıysa df['Product_Group'] kullanmak daha doğru sonuç verebilir.
df[['Main_Category', 'Subcategory']] = df['Product_Group'].apply(akilli_eslestir)

# Sonucu Kontrol Etme (Hangi ürünlerin eşleşemediğini görmek için)
eslesmeyenler = df[df['Main_Category'] == 'Diğer']
print(f"Eşleşmeyen ürün sayısı: {len(eslesmeyenler)}")
if len(eslesmeyenler) > 0:
    print("\nEşleşemeyen ilk 10 ürün:")
    print(eslesmeyenler[['Product_Group']].head(10))

# Eşleşenlerden bir örnek
print("\nEşleşen ürünler örneği:")
print(df[['Product_Group', 'Main_Category', 'Subcategory']].sample(10, random_state=15))

Eşleşmeyen ürün sayısı: 174

Eşleşemeyen ilk 10 ürün:
                         Product_Group
168             Ankastre Mikrodalgalar
188             Gazli Cam Tablali Ocak
189             Gazli Cam Tablali Ocak
190        Entegre Davlumbazli Ocaklar
191        Entegre Davlumbazli Ocaklar
192               Vitroseramik Ocaklar
193  Gazli Inoks Metal Tablali Ocaklar
194             Gazli Cam Tablali Ocak
195                     Domino Ocaklar
196  Gazli Inoks Metal Tablali Ocaklar

Eşleşen ürünler örneği:
                                      Product_Group             Main_Category  \
94                           12 Kg Camasir Makinesi                Beyaz Eşya   
482                                        Kulaklik                Elektronik   
295                     Fitfry Buhar Destekli Firin                Beyaz Eşya   
434                                        Hoparlor                Elektronik   
1015  Kucuk Ev Aletleri Yedek Parca Ve Aksesuarlari         Küçük Ev Aletleri   
262   

In [13]:
df.sample(5, random_state=15)

,urun_linki,urun_adi,fiyat,teknik_ozellikler,enerji_sinifi,favori_sayisi,yorum_sayisi,renk_secenekleri,urun_ozellikleri,yorumlar,Product_Group,Main_Category,Subcategory
94,https://www.beko.com.tr/12-kg-camasir-makinesi...,CMX 12140 12 Kg Çamaşır Makinesi,39.959 TL,"{""Kapasite"": ""12 kg"", ""Maksimum Sıkma Devri"": ...",A,120.0,NaN,[],"{""Kapasite"": ""12 kg"", ""Maksimum Sıkma Devri"": ...","[{""yildiz"": ""5"", ""baslik"": ""F/P ÜRÜNÜ"", ""sahip...",12 Kg Camasir Makinesi,Beyaz Eşya,Çamaşır Makinesi
482,https://www.beko.com.tr/kulaklik/jr310bt-bluet...,JR310BT Bluetooth ÇocukKulaklığı OEMavi Kulaklık,2.190 TL,"{""Kulaklık Tipi"": ""Kablosuz kulak üstü kulaklı...",NaN,0.0,NaN,[],"{""Kulaklık Tipi"": ""Kablosuz kulak üstü kulaklı...",[],Kulaklik,Elektronik,Ses& Görüntü Sistemleri
295,https://www.beko.com.tr/fitfry-buhar-destekli-...,BFC 450 A FitFry Buhar Destekli Fırın,59.405 TL,"{""Motor Teknolojisi"": ""AeroPerfect"", ""Homewhiz...",A+,5.0,NaN,[],"{""Motor Teknolojisi"": ""AeroPerfect"", ""Homewhiz...",[],Fitfry Buhar Destekli Firin,Beyaz Eşya,Fırın
434,https://www.beko.com.tr/hoparlor/band-bt-hopar...,Grundig Band BT Hoparlör Yeşil Hoparlör,2.099 TL,"{""Ses Çıkış Gücü"": ""2x 2,5 W"", ""Çalma Süresi"":...",NaN,2.0,NaN,[],"{""Ses Çıkış Gücü"": ""2x 2,5 W"", ""Çalma Süresi"":...",[],Hoparlor,Elektronik,Ses& Görüntü Sistemleri
1015,https://www.beko.com.tr/kucuk-ev-aletleri-yede...,Bıçak Grubu Blender Küçük Ev Aletleri Yedek Pa...,475 TL,{},NaN,1.0,1,[],{},[],Kucuk Ev Aletleri Yedek Parca Ve Aksesuarlari,Küçük Ev Aletleri,Küçük Ev Aletleri Seti


In [14]:
# Tüm sütunları gez ve boş liste/sözlük yapılarını NaN yap
for col in df.columns:
    df[col] = df[col].apply(lambda x: np.nan if str(x).strip() in ['[]', '{}', ''] else x)

# Fiyat
# 1. 'TL' yazısını ve etrafındaki her türlü boşluğu (\n dahil) regex ile temizle
df['fiyat'] = df['fiyat'].str.replace(r'\s*TL', '', regex=True)

# 2. Binlik ayracı olan noktayı (.) tamamen sil (örn: 39.959 -> 39959)
df['fiyat'] = df['fiyat'].str.replace('.', '', regex=False)

# 3. Eğer başka satırlarda kuruş varsa (örn: 475,50), virgülü ondalık formata (.) çevir
df['fiyat'] = df['fiyat'].str.replace(',', '.', regex=False)

# 4. Temizlenmiş metni artık güvenle float tipine dönüştür
df['fiyat'] = pd.to_numeric(df['fiyat'], errors='coerce')


# Bilimsel gösterimi kapat, sayıları virgülden sonra 2 basamaklı normal formatta göster
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Favori Sayısı
# Metinleri temizle ve sayıya çevir
df['favori_sayisi'] = pd.to_numeric(df['favori_sayisi'], errors='coerce')

# NaN'ları bozmadan tam sayıya çevir (Büyük 'I' harfine dikkat!)
df['favori_sayisi'] = df['favori_sayisi'].astype('Int64')

# NumPy için bilimsel gösterimi kapat (suppress=True)
np.set_printoptions(suppress=True)

# Çıktıyı tekrar kontrol et
print(df['fiyat'].unique())
print(df['favori_sayisi'].unique())

[ 88409.         nan  15159.    57969.    22189.    34829.    19749.
  59129.    28279.    18899.    42409.    48579.    22989.    47839.
  34129.    31389.    39449.    35829.    50489.    36029.    33839.
  36809.    37769.    33869.    40619.    39809.    29489.    38639.
  40259.    32189.    35279.    46799.    39959.    37649.    41069.
  38489.    24479.    25979.    28079.    23729.    34859.    24029.
  33569.    29699.    21179.    23579.    26549.    36149.    30989.
  36869.    23369.    21089.    27899.    27209.    21869.    38999.
  39479.    40379.    39719.    45329.    40799.    38249.    35699.
  23999.    44129.    41849.    43229.    43499.    39749.    37139.
  40499.     8739.    28276.    46466.    15363.    12599.    28734.
  40039.    27982.    45093.    15120.    14700.     9540.    12600.
   5010.     9000.    10260.     5899.     7469.     6599.    11217.
   8119.    15577.    31701.    21499.    12999.     7749.     9949.
  12399.    16849.      525.      

In [15]:
df.head()

,urun_linki,urun_adi,fiyat,teknik_ozellikler,enerji_sinifi,favori_sayisi,yorum_sayisi,renk_secenekleri,urun_ozellikleri,yorumlar,Product_Group,Main_Category,Subcategory
0,https://www.beko.com.tr/no-frost-buzdolabi/683...,683616 EI Yapay Zeka Teknolojili No Frost Buzd...,88409.00,"{""Dondurucu Bölme Teknolojisi"": ""Sabit Set Değ...",C,104,NaN,NaN,"{""Dondurucu Bölme Teknolojisi"": ""Sabit Set Değ...","[{""yildiz"": ""5"", ""baslik"": ""Beko Teşekkürler"",...",No Frost Buzdolabi,Beyaz Eşya,Buzdolabı
1,https://www.beko.com.tr/no-frost-buzdolabi/678...,678551 EGC AI Fit No Frost Buzdolabı,NaN,"{""76 cm"": ""Derinlik"", ""78 cm"": ""Genişlik"", ""18...",D,4,NaN,NaN,NaN,NaN,No Frost Buzdolabi,Beyaz Eşya,Buzdolabı
2,https://www.beko.com.tr/no-frost-buzdolabi/678...,678551 EBC AI Fit Beyaz Cam Yapay Zeka Teknolo...,NaN,"{""76 cm"": ""Derinlik"", ""78 cm"": ""Genişlik"", ""18...",D,7,NaN,NaN,NaN,"[{""yildiz"": ""5"", ""baslik"": ""güzel"", ""sahip_adi...",No Frost Buzdolabi,Beyaz Eşya,Buzdolabı
3,https://www.beko.com.tr/no-frost-buzdolabi/678...,678551 ESC Siyah Cam No Frost Buzdolabı,NaN,"{""Dondurucu Bölme Teknolojisi"": ""Sabit Set Değ...",D,41,NaN,NaN,"{""Dondurucu Bölme Teknolojisi"": ""Sabit Set Değ...","[{""yildiz"": ""5"", ""baslik"": """", ""sahip_adi"": ""A...",No Frost Buzdolabi,Beyaz Eşya,Buzdolabı
4,https://www.beko.com.tr/no-frost-buzdolabi/670...,670475 IEBC AI Beyaz Cam No Frost Buzdolabı,NaN,"{""75 cm"": ""Derinlik"", ""70 cm"": ""Genişlik"", ""18...",E,4,NaN,NaN,NaN,NaN,No Frost Buzdolabi,Beyaz Eşya,Buzdolabı
